In [6]:
# Author: Niko Bleidistel
# last change: 2026-08-06

# Package Import

In [7]:
from pathlib import Path 
from os import makedirs
import sys
import importlib
import re

import pandas as pd
import numpy as np
import math

import mph
import matplotlib as mpl
import matplotlib.pyplot as plt

In [8]:
PYTHON_HELPER_FOLDER = Path(r"py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import comsol_data_export as cde
import comsol_data_plotting as cdp
import plot_functions as pfs
import time_logging as tl

# reload custom packages (for each execution) to reflect recent changes
_ = importlib.reload(cde)
_ = importlib.reload(cdp)
_ = importlib.reload(pfs)
_ = importlib.reload(tl)

# PATHS

In [ ]:
# INPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\06_mfco_assymmetry")
# OUTPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Script Outputs\000_New_Output")

MAIN_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\10_bachelor_thesis_models_use_terminals")
INPUT_FOLDER = MAIN_FOLDER / "Solved model versions"
OUTPUT_FOLDER = MAIN_FOLDER / "Test Output"

makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

# INITIALIZE

In [ ]:
# initialize time logging
_ = tl.initialize_time_log(OUTPUT_FOLDER / 'time_log.csv')

# initialize COMSOL client (server)
client = mph.start()

# SIMULATE

## Constants

In [ ]:
EXPORT_DICT = {
    "mf.normB": "Magnetic flux density, norm [T]",
    "mf.Bx": "Magnetic flux density, x-component [T]", 
    "mf.By": "Magnetic flux density, y-component [T]", 
    "mf.Bz": "Magnetic flux density, z-component [T]",
    "T": "Temperature [K]",
}
EXPORT_PARAMS = list(EXPORT_DICT.keys())
EXPORT_DESCRIPTION = list(EXPORT_DICT.values())

CONDUCTOR_EXPORT_DICT = {
    "V": "Electric potential [V]",
    "ec.normJ": "Current density, norm [A/m^2]",
    "ec.Jx": "Current density, x-component [A/m^2]",
    "ec.Jy": "Current density, y-component [A/m^2]",
    "ec.Jz": "Current density, z-component [A/m^2]",
}
CONDUCTOR_EXPORT_PARAMS = list(CONDUCTOR_EXPORT_DICT.keys())
CONDUCTOR_EXPORT_DESCRIPTION = list(CONDUCTOR_EXPORT_DICT.values())

## Simulation Order

In [ ]:
GROUP_1 = [
    "01_01_a-Round spiral",
    "01_01_b-Rectangular spiral",
    "01_02_a-Grid",
    "01_03_a-Round spiral combined with grid",
    "01_03_b-Rectangular spiral combined with grid",
]

GROUP_2 = [
    "01_00_d-high resolution planes and increasing areas",
    "01_00_c-high resolution planes",
    "01_00_b-high resolution cuboid",
    "01_00_a-auto mesh"
]

GROUP_3 = [
    "02_00_a-H design with minimized insulator",
    "02_00_b-H design with full insulator",
]

## Simulate

### Group 1

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group1"

    for g1 in GROUP_1:
        modelfile = input_folder / f"{g1}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,

                    # for sweeps
                    iteration_number = None,
                    model = None,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

### Group 2

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group2"

    for g2 in GROUP_2:
        modelfile = input_folder / f"{g2}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, "-1*insulator_height"),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height-1*insulator_height"),

                    Homogeneity_point1 = (0.0, "+0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_point2 = (0.0, "-0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_point2 = ("-0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,

                    # for sweeps
                    iteration_number = None,
                    model = None,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

### Group 3

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group3"

    for g3 in GROUP_3:
        modelfile = input_folder / f"{g3}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, "-1*insulator_height"),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height-1*insulator_height"),

                    Homogeneity_point1 = (0.0, "+0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_point2 = (0.0, "-0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_point2 = ("-0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = "-1*epilayer_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,

                    # for sweeps
                    iteration_number = None,
                    model = None,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

## SWEEPS
- H design: 4 strom konfis
- Round Spiral: number turns = 15, 25, 35
- Spirals + Grid: Voltage Distributions = 0, 45, 67, 90 degree (left out lines = 0) 
- Grid: Left out Lines = 0, 1, 2
    - in 3-degree increments from 0 to 90 degrees for each 'left out lines'

### Group 3

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group3"

    for g3 in GROUP_3:
        modelfile = input_folder / f"{g3}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameter = [
                        "I_conductor_A_terminal",
                        "I_conductor_B_minus_terminal",
                        "I_conductor_B_plus_terminal",
                    ],
                    sweep_values = [
                        [0e-6, 0e-6, 5e-6, 10e-6],
                        [10e-6, 10e-6, 10e-6, 10e-6],
                        [10e-6, -10e-6, -10e-6, -10e-6],
                    ],

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, "-1*insulator_height"),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height-1*insulator_height"),

                    Homogeneity_point1 = (0.0, "+0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_point2 = (0.0, "-0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_point2 = ("-0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = "-1*epilayer_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

### Group 1

In [ ]:
SWEEP_GROUP1_1 = [
    "01_01_a-Round spiral",
]
SWEEP_GROUP1_3 = [
    "01_03_a-Round spiral combined with grid",
    "01_03_b-Rectangular spiral combined with grid",
]
SWEEP_GROUP1_2 = [
    "01_02_a-Grid",
]

#### Group 1.1

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group1"

    for g1_1 in SWEEP_GROUP1_1:
        modelfile = input_folder / f"{g1_1}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep1"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameter = [
                        "N_spiral_turns",
                    ],
                    sweep_values = [
                        [15, 25, 35],
                    ],

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

#### Group 1.3

In [ ]:
SWEEP_DICT_1_3 = cde.get_voltage_sweep_dict(
    angles = [0, 45, 67, 90],
    magnitude = 10e-6,
    conductor_grid_length = 60e-6,
)

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group1"

    for g1_3 in SWEEP_GROUP1_3:
        modelfile = input_folder / f"{g1_3}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep3"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameter = list(SWEEP_DICT_1_3.keys()),
                    sweep_values = list(SWEEP_DICT_1_3.values()),

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

#### Group 1.2

In [ ]:
angles = list(range(0, 91, 3))  # angles from 0 to 90 degrees in steps of 3 degrees
VOLTAGE_DICT_1_2 = cde.get_voltage_sweep_dict(
    angles = angles,
    magnitude = 10e-6,
    conductor_grid_length = 60e-6,
) # dictionary: for each terminal, a list of the corresponding voltage values for each angle in the sweep

parameters_1_2 = ["left_out_lines"] + list(VOLTAGE_DICT_1_2.keys()) # list of parameters for the sweep, including the left_out_lines parameter and the voltage parameters for each terminal
left_out_lines_values = [0, 1, 2]

values_1_2 = [] # list of lists, where each inner list contains the values for the parameters in parameters_1_2 for a combination

for i in range(len(left_out_lines_values)):
    for k in range(len(list(VOLTAGE_DICT_1_2.values())[0])):
        row = [left_out_lines_values[i]] + [VOLTAGE_DICT_1_2[key][k] for key in VOLTAGE_DICT_1_2.keys()]
        values_1_2.append(row)


In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "Group1"

    for g1_2 in SWEEP_GROUP1_2:
        modelfile = input_folder / f"{g1_2}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep2"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameter = parameters_1_2,
                    sweep_values = values_1_2,

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

# END

In [ ]:
tl.log_message("Reached the end of the script.")